In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import CLIPModel, CLIPProcessor

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error



2026-05-06 14:22:51.358807: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778077371.371624      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778077371.375394      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-06 14:22:51.390136: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
# Paths
base_dir = "/kaggle/input/petfinder-pawpularity-score"
train_csv = os.path.join(base_dir, "train.csv")
test_csv = os.path.join(base_dir, "test.csv")

train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

print(f"Train samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")




Train samples: 8920
Test samples: 992


In [3]:
class ImageDataset(Dataset):
    """Returns CLIP‑compatible image tensors."""

    def __init__(self, df, root_dir, processor, split="train"):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.processor = processor
        self.split = split  # 'train' or 'test'

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.loc[idx, "Id"]
        img_path = os.path.join(self.root_dir, self.split, f"{img_id}.jpg")
        image = Image.open(img_path).convert("RGB")
        # processor returns a dict; we only need pixel_values
        processed = self.processor(images=image, return_tensors="pt")
        # squeeze batch dimension to get a single tensor
        return processed["pixel_values"].squeeze(0)




In [4]:
# Load CLIP model & processor (CPU version to avoid GPU driver issues)
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name)
processor = CLIPProcessor.from_pretrained(model_name)

device = torch.device("cpu")
model = model.to(device)
model.eval()



config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [5]:
batch_size = 64

train_dataset = ImageDataset(train_df, base_dir, processor, split="train")
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=False, num_workers=2
)

test_dataset = ImageDataset(test_df, base_dir, processor, split="test")
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=2
)




In [6]:
def extract_features(dataloader):
    """Run CLIP encoder on all images in a dataloader."""
    all_feats = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting features"):
            # batch shape: (B, C, H, W)
            batch = batch.to(device)
            feats = model.get_image_features(pixel_values=batch)
            all_feats.append(feats.cpu().numpy())
    return np.concatenate(all_feats, axis=0)


X_train = extract_features(train_loader)  # shape (num_train, 512)
X_test = extract_features(test_loader)  # shape (num_test, 512)

print("Feature shapes:", X_train.shape, X_test.shape)




Extracting features: 100%|██████████| 16/16 [00:34<00:00,  2.16s/it]

Feature shapes: (8920, 512) (992, 512)


In [7]:
# Scale features (fit on training set only)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Target values (original scale 0‑100)
y_train = train_df["Pawpularity"].values



In [8]:
# Train a simple SVR (CPU, same hyper‑parameters as original attempt)
svr = SVR(C=16.0, kernel="rbf", degree=3, max_iter=400000)
svr.fit(X_train, y_train)

# Optional: compute training RMSE to see where we stand
train_pred = svr.predict(X_train)
train_rmse = mean_squared_error(y_train, train_pred, squared=False)
print(f"Training RMSE: {train_rmse:.4f}")



Training RMSE: 13.3627


In [9]:
# Predict on test set
test_pred = svr.predict(X_test)

# Clamp predictions to the valid range [0, 100]
test_pred = np.clip(test_pred, 0, 100)



In [10]:
submission = pd.DataFrame({"Id": test_df["Id"], "Pawpularity": test_pred})
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")

Submission saved to submission.csv
